In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim

from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)


def start(mission: MissionBase):
    sim = CarSim(prop, mission)
    complete = sim.run()
    if complete:
        SimDrawer(sim).show()
        goal_cnt = sim.history.goal_cnt[-1]
        goals = len(mission.goals)
        goaled = goal_cnt == goals
        return goaled
    else:
        return False

In [ ]:
class Mission4(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=80)
        self.goals = [
            GoalCircle((2.2, 1.6), 0.2, should_stop=True),
            GoalCircle((2.5, 0.0), 0.2, should_stop=False),
        ]
        self.initial_xy = (2.5, 0.0)
        self.random_d_xy = (0.1, 0.1)
        self.random_d_yaw_deg = 5
        self.set_signs(
            [
                Sign(x=2.2, y=1.6, name="original"),
                ### 標識を追加するにはここから下を書き換える
                Sign(x=2.0, y=1.0, name="stop"),
                Sign(x=1.4, y=0.9, name="warn"),
                ### 標識を追加するにはここより上を書き換える
            ]
        )

    @staticmethod
    def command_func(alive, *, move, rotate, wait, search, auto, **kwargs):
        ######## ここから下にプログラムを書こう
        mode = 0
        while alive():
            if mode == 0:
                # コース内で自動走行し、標識(original)を見つけた場合に標識の真上に行き１秒停車するモード
                pos = search(name="original")
                if pos is None:
                    auto(v=0.2)
                else:
                    if pos.theta > 5:
                        rotate(w=45)
                    elif pos.theta < -5:
                        rotate(w=-45)
                    else:
                        move(v=0.2)
                        move(v=0.2, t=pos.x / 0.2)
                        wait()
                        move(v=0, t=1)
                        wait()
                        mode = 1
            elif mode == 1:
                # 左回転し、標識(stop)を見つけた場合に標識の真上に行くモード
                pos = search(name="stop")
                if pos is None:
                    rotate(w=45)
                else:
                    if pos.theta > 5:
                        rotate(w=45)
                    elif pos.theta < -5:
                        rotate(w=-45)
                    else:
                        move(v=0.2)
                        move(v=0.2, t=pos.x / 0.2)
                        wait()
                        mode = 2
            elif mode == 2:
                # 右回転し、標識(warn)を見つけた場合に標識の方を向いた後で自動走行を開始するモード
                pos = search(name="warn")
                if pos is None:
                    rotate(w=-45)
                else:
                    rotate(w=pos.theta, t=1)
                    wait()
                    auto(v=0.2)
                    mode = 0

        ######## ここより上にプログラムを書こう


"成功" if start(Mission4()) else "失敗"

In [ ]:
# 複数回実行して必ず成功するかのチェック
CHECK_CNT = 10
ok = 0
for i in range(CHECK_CNT):
    mission = Mission4()
    sim = CarSim(prop, mission)
    complete = sim.run()
    if complete:
        goal_cnt = sim.history.goal_cnt[-1]
        goals = len(mission.goals)
        goaled = goal_cnt == goals
        if goaled:
            ok += 1
        else:
            SimDrawer(sim).show()
            break
        print("-------------------------------OK:", ok)